# BDA Practica 2 — Model Comparison and Explainability

Three predictive pipelines are trained on the integrated dataset:

1. **Integrated Core** — Logistic Regression / Random Forest on a small set of widely-available indicators (age, BMI, gender, risk-factor flags).
2. **Integrated Enriched** — same model family with an enriched feature set that adds clinical variables (BP, max heart rate, mental/physical unhealthy days, etc.).
3. **KG Embedding Classifier** — Logistic Regression / Random Forest trained on **graph-derived features** built by holding out outcome aggregate nodes, embedding the remaining typed RDF adjacency with TruncatedSVD, and then composing per-record feature vectors from dataset, population-group, and indicator node embeddings (plus interactions).

This is the **second analytical pipeline** required by P2: ML over KG embeddings. The graph-based exploration is in `02_knowledge_graph_exploration.ipynb`.

Run the full pipeline (`python run_all_pipeline.py --skip-landing --strict`) before opening this notebook.

## 1. Setup and Reports

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def locate_project_root(start: Path) -> Path:
    current = start.resolve()
    for _ in range(8):
        if (current / 'run_all_pipeline.py').exists():
            return current
        current = current.parent
    raise FileNotFoundError('Could not locate project root')

PROJECT_ROOT = locate_project_root(Path.cwd())
REPORTS_DIR = PROJECT_ROOT / 'Part5_Analysis_zone' / 'reports'

integrated_core = json.loads((REPORTS_DIR / 'integrated_core_report.json').read_text(encoding='utf-8'))
integrated_enriched = json.loads((REPORTS_DIR / 'integrated_enriched_report.json').read_text(encoding='utf-8'))
kg_embedding = json.loads((REPORTS_DIR / 'kg_embedding_report.json').read_text(encoding='utf-8'))
summary = json.loads((REPORTS_DIR / 'summary_report.json').read_text(encoding='utf-8'))

MODELS = [
    ('Integrated Core',     integrated_core,     '#3B7DD8'),
    ('Integrated Enriched', integrated_enriched, '#218A4D'),
    ('KG Embedding ML',     kg_embedding,        '#C24B65'),
]
for name, report, _ in MODELS:
    print(f'{name:22s} selected model = {report["selected_model"]}')

## 2. Test-set Performance Comparison

We compare the three pipelines on the same test split semantics: stratified by outcome, decision threshold tuned for F1 on the training fold. Note that the KG embedding pipeline uses a stratified record-level sample over the same integrated dataset, so the comparison is fair in distribution but uses different features (graph node embeddings vs raw indicators).

In [ ]:
metrics_rows = []
for name, report, _ in MODELS:
    m = report['test_metrics']
    metrics_rows.append({
        'pipeline': name,
        'selected_model': report['selected_model'],
        'accuracy':  m.get('accuracy'),
        'precision': m.get('precision'),
        'recall':    m.get('recall'),
        'f1':        m.get('f1'),
        'roc_auc':   m.get('roc_auc'),
        'pr_auc':    m.get('pr_auc'),
    })
metrics_df = pd.DataFrame(metrics_rows)
metrics_df

In [ ]:
to_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
values = metrics_df.set_index('pipeline')[to_plot]
fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
values.plot(kind='bar', ax=ax, width=0.78, colormap='viridis')
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Test-set metrics per pipeline')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.32), ncol=6, frameon=False)
plt.xticks(rotation=0)
plt.show()

## 3. Confusion Matrices

Confusion matrices on the held-out test set give a clearer picture of trade-offs between false positives and false negatives. This is particularly important in heart-disease screening where false negatives carry a high cost.

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(15, 4.6), constrained_layout=True)
for ax, (name, report, _) in zip(axes, MODELS):
    cm = np.array(report['test_metrics']['confusion_matrix'])
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(f'{name}\n{report["selected_model"]}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['negative', 'positive'])
    ax.set_yticklabels(['negative', 'positive'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center', color='black')
plt.show()

## 4. Cross-Validation Stability

Comparing the cross-validation mean (and standard deviation) of the candidate models for each pipeline gives a sense of stability. Wider gaps between the chosen model and the alternative imply that the metric used for selection is meaningful.

In [ ]:
cv_rows = []
for name, report, _ in MODELS:
    candidate_models = report.get('candidate_models', {})
    for model_name, scores in candidate_models.items():
        cv_rows.append({
            'pipeline': name,
            'model':    model_name,
            'cv_score_mean': scores.get('cv_roc_auc_mean', scores.get('cv_pr_auc_mean')),
            'cv_score_std':  scores.get('cv_roc_auc_std',  scores.get('cv_pr_auc_std')),
            'scoring':       'roc_auc' if 'cv_roc_auc_mean' in scores else 'pr_auc',
        })
cv_df = pd.DataFrame(cv_rows)
cv_df

## 5. Feature Importance — Tabular Models

The two integrated pipelines store the top ranked features that drove the chosen classifier. Random Forest exposes Gini importance; Logistic Regression exposes the absolute value of the standardized coefficients. Both are appropriate proxies for relative feature contribution.

In [ ]:
def plot_feature_importance(report: dict, title: str, ax, color: str = '#4F8DCB', top_n: int = 12) -> None:
    items = report.get('top_feature_importance', [])[:top_n][::-1]
    if not items:
        ax.set_title(f'{title}: no importance available')
        return
    labels = [item['feature'] for item in items]
    values = [item['importance'] for item in items]
    ax.barh(labels, values, color=color)
    ax.set_title(title)
    ax.set_xlabel(items[-1].get('importance_type', 'importance'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)
plot_feature_importance(integrated_core, 'Integrated Core — Top features', axes[0], '#3B7DD8')
plot_feature_importance(integrated_enriched, 'Integrated Enriched — Top features', axes[1], '#218A4D')
plt.show()

## 6. KG Embedding ML — Structural Inspection

The KG embedding classifier predicts the same target (heart disease outcome) using **only graph-derived features**:

- A *dataset* embedding (16 dimensions).
- A *population-group* embedding (16 dimensions, encodes the age × gender concept).
- Mean embeddings of the *observed*, *risk-factor*, and *protective-factor* indicators connected to the record.
- Interaction features (`dataset ⊙ group`, `|dataset − group|`).
- Three count statistics (observed/risk/protective indicators).

Heart-disease outcome aggregate nodes are **held out** while embeddings are generated to avoid label leakage.

In [ ]:
print('Graph:')
for key, value in kg_embedding['graph'].items():
    print(f'  {key:30s} {value:,}' if isinstance(value, int) else f'  {key:30s} {value}')
print('Samples:')
for key, value in kg_embedding['samples'].items():
    if isinstance(value, dict):
        print(f'  {key}:')
        for k2, v2 in value.items():
            print(f'    {k2}: {v2}')
    else:
        print(f'  {key:30s} {value}')
print(f'Selected model: {kg_embedding["selected_model"]}')
print(f'Decision threshold: {kg_embedding["decision_threshold"]:.4f}')

In [ ]:
embeddings_path = REPORTS_DIR / 'kg_node_embeddings.csv'
if embeddings_path.exists():
    emb = pd.read_csv(embeddings_path)
    print(f'Loaded {len(emb):,} node embeddings (dim={emb.shape[1]-1})')
    emb.head()

### 6.1 2D Projection of the Node Embeddings

A simple 2D projection (using the first two embedding dimensions) highlights how the SVD over the typed RDF adjacency separates datasets, indicators and population groups in the latent space.

In [ ]:
if embeddings_path.exists():
    BASE = 'https://example.org/bda/health-risk/'

    def node_kind(name: str) -> str:
        if name.startswith(BASE + 'dataset/'):
            return 'dataset'
        if name.startswith(BASE + 'indicator/'):
            return 'indicator'
        if name.startswith(BASE + 'population-group/'):
            return 'population_group'
        if name.startswith(BASE + 'age-group/'):
            return 'age_group'
        if name.startswith(BASE + 'gender/'):
            return 'gender'
        if name.startswith(BASE + 'outcome/'):
            return 'outcome'
        if name.startswith(BASE + 'aggregate-measurement/'):
            return 'aggregate'
        if '#predicate' in name:
            return 'predicate'
        return 'other'

    emb_viz = emb.copy()
    emb_viz['kind'] = emb_viz['node'].map(node_kind)
    interesting = ['dataset', 'indicator', 'population_group', 'outcome', 'age_group', 'gender']
    emb_viz = emb_viz[emb_viz['kind'].isin(interesting)]

    fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
    colors = {
        'dataset': '#C24B65',
        'indicator': '#218A4D',
        'population_group': '#B8B8B8',
        'outcome': '#5C8FA1',
        'age_group': '#7A6CC6',
        'gender': '#E59B0F',
    }
    sizes = {
        'dataset': 110, 'indicator': 90, 'outcome': 110,
        'age_group': 70, 'gender': 70, 'population_group': 14,
    }
    for kind, group in emb_viz.groupby('kind'):
        ax.scatter(group['kg_emb_00'], group['kg_emb_01'], s=sizes[kind], alpha=0.7,
                   c=colors[kind], label=kind, edgecolors='white', linewidths=0.4)
    ax.set_xlabel('KG embedding dim 0')
    ax.set_ylabel('KG embedding dim 1')
    ax.set_title('Node embeddings projected to the first two SVD dimensions')
    ax.legend(loc='best', frameon=False)
    plt.show()

### 6.2 Label distribution of the KG-embedding training set

In [ ]:
audit_path = REPORTS_DIR / 'kg_embedding_training_data.csv'
if audit_path.exists():
    audit = pd.read_csv(audit_path)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    audit['target'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='#5C8FA1')
    axes[0].set_title('Target distribution (0 = neg, 1 = pos)')
    axes[0].set_xlabel('target')
    audit['source_dataset'].value_counts().plot(kind='bar', ax=axes[1], color='#C24B65')
    axes[1].set_title('Source dataset distribution')
    plt.xticks(rotation=20, ha='right')
    plt.show()
    audit.describe(include='all').T

## 7. Interpretation

- The **Integrated Enriched** baseline tends to be the strongest classifier because it has access to clinical variables (BP, max heart rate, mental/physical unhealthy days). It is the natural reference point for any new approach.
- The **KG Embedding** model has a strictly different feature space: it only sees graph-derived signals (dataset, population group, observed/risk indicator buckets and their interactions). It demonstrates that the KG carries enough signal to be predictive on its own, which is the value proposition of the P2 graph-embedding pipeline.
- The KG embedding model also generalises well because the embeddings are computed without any direct connection to outcome aggregate nodes (those are held out during embedding generation), reducing label leakage.
- Looking at the **feature importance** tables, the strongest tabular features are typically `age_years_proxy`, BMI and the risk-factor flags — consistent with the SPARQL outcome rate analysis in `02_knowledge_graph_exploration.ipynb` which ranks population groups primarily by age and high blood pressure.
- Confusion matrices reveal where models differ on the precision/recall trade-off; this is an actionable signal for downstream decisions (e.g., screening recall vs alert burden).

## 8. Reproducibility

All artefacts live under `Part5_Analysis_zone/`:

- `models/integrated_core_model.pkl`
- `models/integrated_enriched_model.pkl`
- `models/kg_embedding_model.pkl`
- `reports/integrated_core_report.json`
- `reports/integrated_enriched_report.json`
- `reports/kg_embedding_report.json`
- `reports/kg_analysis_report.json`
- `reports/summary_report.json`
- `reports/kg_node_embeddings.csv`
- `reports/kg_embedding_training_data.csv`

Re-run with `python run_all_pipeline.py --skip-landing --strict`.